In [3]:
import pandas as pd

# read jsonl file
data = pd.read_json('data/train.jsonl', lines=True)

### Test Simple Text to Text Retrieval

In [ ]:
import tqdm
# 創建一個新欄位 image_caption label 把相同 paper_id 的 caption 合併在一起，用空格隔開
data['image_caption_label'] = data.groupby('paper_id')['image_caption'].transform(lambda x: ','.join(x))

# 用 embedding model 從 image_caption_label 中找出 跟 query 最相似的 caption
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('intfloat/e5-large-v2')

for index, row in tqdm.tqdm(data.iterrows(), total=len(data)):
    query = row['query']
    caption_label = row['image_caption_label']
    
    # 將 caption_label 分割成多個句子
    captions = caption_label.split(',')
    
    # 對 query 和每個 caption 分別生成 embedding
    query_embedding = model.encode(query, convert_to_tensor=True)
    caption_embeddings = model.encode(captions, convert_to_tensor=True)
    
    # 計算 query 與每個 caption 的相似度
    cos_sim = util.pytorch_cos_sim(query_embedding, caption_embeddings)
    
    # 找到最相似的前三個 captions
    top_indices = cos_sim.topk(3).indices.squeeze(0).tolist()
    top_captions = [captions[i] for i in top_indices]
    
    # 將結果存入 DataFrame，將列表轉換為字符串格式
    data.at[index, 'top_captions'] = ','.join(top_captions)

# 比對 image_caption 是否在前三個最相似的 captions 中
data['caption_match'] = data.apply(lambda x: x['image_caption'] in x['top_captions'].split(','), axis=1)

# 計算 caption_match 的準確率
accuracy = data['caption_match'].mean()
print(f'Caption Match Accuracy: {accuracy:.4f}')

In [ ]:
import tqdm
# 創建一個新欄位 image_path label 把相同 paper_id 的 image_path 合併在一起，用空格隔開
data['image_path_label'] = data.groupby('paper_id')['image_path'].transform(lambda x: ','.join(x))

# 使用 CLIP 模型從 image_path_label 中找出 跟 query 最相似的 image
from transformers import CLIPProcessor, CLIPModel
import torch

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

for index, row in tqdm.tqdm(data.iterrows(), total=len(data)):
    query = row['query']
    image_path_label = row['image_path_label']
    
    # 將 image_path_label 分割成多個路徑
    image_paths = image_path_label.split(',')
    
    images = []
    for image_path in image_paths:
        image = Image.open(image_path).convert("RGB")
        images.append(image)
    
    # 對 query 和每個 image 分別生成 embedding
    inputs = processor(text=[query]*len(images), images=images, return_tensors="pt", padding=True)
    outputs = model(**inputs)
    
    text_embeddings = outputs.text_embeds
    image_embeddings = outputs.image_embeds
    
    # 計算 query 與每個 image 的相似度
    cos_sim = torch.nn.functional.cosine_similarity(text_embeddings, image_embeddings)
    
    # 找到最相似的前三個 images
    top_indices = torch.topk(cos_sim, 3).indices.tolist()
    top_image_paths = [image_paths[i] for i in top_indices]
    
    # 將結果存入 DataFrame，將列表轉換為字符串格式
    data.at[index, 'top_image_paths'] = ','.join(top_image_paths)
# 比對 image_path 是否在前三個最相似的 image_paths 中
data['image_path_match'] = data.apply(lambda x: x['image_path'] in x['top_image_paths'].split(','), axis=1) 

# 計算 image_path_match 的準確率
accuracy = data['image_path_match'].mean()
print(f'Image Path Match Accuracy: {accuracy:.4f}')

In [32]:
# read jsonl file
test_query = pd.read_json('data/test.jsonl', lines=True)
test_image = pd.read_json('data/test_images.jsonl', lines=True)

In [33]:
# 合併相同 paper_id 的行
test_image_grouped = test_image.groupby('paper_id').agg(lambda x: list(x)).reset_index()

# 用 paper_id 當作 key
test_image_dict = test_image_grouped.set_index('paper_id').to_dict(orient='index')

In [34]:
# 根據 test_image_dict 中的 key 把 test_image_dict 的值 放入 test_query
test_query['image_data'] = test_query['paper_id'].map(test_image_dict)

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import tqdm

model = SentenceTransformer('intfloat/e5-large-v2')

submission_results = []

for index, row in tqdm.tqdm(test_query.iterrows(), total=len(test_query)):
    query = row['query']
    # image_data_for_paper will be a dictionary like {'image_id': [...], 'image_caption': [...]} for the current paper
    image_data_for_paper = row['image_data']

    # Encode the query
    query_embedding = model.encode(query, convert_to_tensor=True)

    image_scores_for_paper = []

    # Iterate through the image_ids and their captions for the current paper
    # Assuming image_data_for_paper['image_id'] and image_data_for_paper['image_caption'] are lists of the same length
    for i in range(len(image_data_for_paper['image_id'])):

        image_id = image_data_for_paper['image_id'][i]
        
        captions_list_for_this_image = image_data_for_paper['image_caption'][i]

        # Ensure captions_list_for_this_image is a list of strings
        if isinstance(captions_list_for_this_image, str):
            captions_list_for_this_image = [captions_list_for_this_image]
        elif not isinstance(captions_list_for_this_image, list) or not all(isinstance(c, str) for c in captions_list_for_this_image):
            continue

        if not captions_list_for_this_image:
            continue

        # Generate embeddings for the captions of the current image
        caption_embeddings = model.encode(captions_list_for_this_image, convert_to_tensor=True)

        # Calculate cosine similarity between the query and each caption of the current image
        cos_sim = util.pytorch_cos_sim(query_embedding, caption_embeddings)

        max_sim_score_for_image = cos_sim.max().item()
        image_scores_for_paper.append((max_sim_score_for_image, image_id))

    image_scores_for_paper.sort(key=lambda x: x[0], reverse=True)

    top_image_ids = []
    seen_image_ids = set()
    for score, img_id in image_scores_for_paper:
        if img_id not in seen_image_ids:
            top_image_ids.append(str(img_id))
            seen_image_ids.add(img_id)
        if len(top_image_ids) == 3:
            break

    while len(top_image_ids) < 3:
        top_image_ids.append("")

    submission_results.append(f"{index+1},{' '.join(top_image_ids[:3])}")

with open('submission.csv', 'w') as f:
    f.write("id,image_id\n")
    f.write("\n".join(submission_results))

print("Submission file generated: submission.csv")

['1611.07718v2-Figure1-1',
 '1611.07718v2-Figure2-1',
 '1611.07718v2-Figure3-1',
 '1611.07718v2-Figure4-1',
 '1611.07718v2-Figure5-1',
 '1611.07718v2-Figure6-1',
 '1611.07718v2-Figure7-1',
 '1611.07718v2-Figure8-1',
 '1611.07718v2-Table2-1',
 '1611.07718v2-Table3-1',
 '1611.07718v2-Table4-1',
 '1611.07718v2-Table5-1']

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import tqdm

model = SentenceTransformer('intfloat/e5-large-v2')

submission_results = []

for index, row in tqdm.tqdm(test_query.iterrows(), total=len(test_query)):
    query = row['query']
    # image_data_for_paper will be a dictionary like {'image_id': [...], 'image_caption': [...]} for the current paper
    image_data_for_paper = row['image_data']

    # Encode the query
    query_embedding = model.encode(query, convert_to_tensor=True)

    image_scores_for_paper = []

    # Iterate through the image_ids and their captions for the current paper
    # Assuming image_data_for_paper['image_id'] and image_data_for_paper['image_caption'] are lists of the same length
    for i in range(len(image_data_for_paper['image_id'])):

        image_id = image_data_for_paper['image_id'][i]
        
        captions_list_for_this_image = image_data_for_paper['image_caption'][i]

        # Ensure captions_list_for_this_image is a list of strings
        if isinstance(captions_list_for_this_image, str):
            captions_list_for_this_image = [captions_list_for_this_image]
        elif not isinstance(captions_list_for_this_image, list) or not all(isinstance(c, str) for c in captions_list_for_this_image):
            continue

        if not captions_list_for_this_image:
            continue

        # Generate embeddings for the captions of the current image
        caption_embeddings = model.encode(captions_list_for_this_image, convert_to_tensor=True)

        # Calculate cosine similarity between the query and each caption of the current image
        cos_sim = util.pytorch_cos_sim(query_embedding, caption_embeddings)

        max_sim_score_for_image = cos_sim.max().item()
        image_scores_for_paper.append((max_sim_score_for_image, image_id))

    image_scores_for_paper.sort(key=lambda x: x[0], reverse=True)

    top_image_ids = []
    seen_image_ids = set()
    for score, img_id in image_scores_for_paper:
        if img_id not in seen_image_ids:
            top_image_ids.append(str(img_id))
            seen_image_ids.add(img_id)
        if len(top_image_ids) == 3:
            break

    while len(top_image_ids) < 3:
        top_image_ids.append("")

    submission_results.append(f"{index+1},{' '.join(top_image_ids[:3])}")

with open('submission.csv', 'w') as f:
    f.write("id,image_id\n")
    f.write("\n".join(submission_results))

print("Submission file generated: submission.csv")

In [ ]:
import pandas as pd
from transformers import CLIPProcessor, CLIPModel
import torch
import tqdm
from PIL import Image

# 初始化 CLIP 模型
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

submission_results = []

for index, row in tqdm.tqdm(test_query.iterrows(), total=len(test_query)):
    query = row['query']
    image_data_for_paper = row['image_data']  # 包含 image_id 和 image_path 的字典列表

    image_scores_for_paper = []

    # 遍歷每張圖片
    for i in range(len(image_data_for_paper['image_id'])):
        image_id = image_data_for_paper['image_id'][i]
        image_path = image_data_for_paper['image_path'][i]

        try:
            # 加載圖像
            image = Image.open(image_path).convert("RGB")

            # 對 query 和圖像生成特徵
            inputs = processor(text=[query], images=[image], return_tensors="pt", padding=True)
            outputs = model(**inputs)

            # 提取文本和圖像的嵌入
            text_embedding = outputs.text_embeds
            image_embedding = outputs.image_embeds

            # 計算 query 與圖像的相似度
            cos_sim = torch.nn.functional.cosine_similarity(text_embedding, image_embedding).item()
            image_scores_for_paper.append((cos_sim, image_id))
        except Exception as e:
            print(f"Error processing image {image_path}: {e}")
            continue

    # 按相似度排序
    image_scores_for_paper.sort(key=lambda x: x[0], reverse=True)

    # 提取前三個最相似的 image_id
    top_image_ids = []
    seen_image_ids = set()
    for score, img_id in image_scores_for_paper:
        if img_id not in seen_image_ids:
            top_image_ids.append(str(img_id))
            seen_image_ids.add(img_id)
        if len(top_image_ids) == 3:
            break

    # 如果不足三個，補空字串
    while len(top_image_ids) < 3:
        top_image_ids.append("")

    # 將結果加入 submission
    submission_results.append(f"{index+1},{' '.join(top_image_ids[:3])}")

# 將結果寫入 submission.csv
with open('submission.csv', 'w') as f:
    f.write("id,image_id\n")
    f.write("\n".join(submission_results))

print("Submission file generated: submission.csv")